# Physics-Informed RIR Generation Framework

> **Notebook goal:** Build an end-to-end generative pipeline that maps macroscopic room parameters to realistic Room Impulse Responses (RIRs), guided by acoustic physics priors.

---

## Phase 1 — Data Pipeline

**Dataset:** HuggingFace [`mandipgoswami/rirmega`](https://huggingface.co/datasets/mandipgoswami/rirmega) (4 000 measured RIRs)

### Tensor Schema

| Tensor | Shape | Contents |
|:-------|:-----:|:---------|
| **X** *(input)* | `[16]` | `[L, W, H, src_x, src_y, src_z, mic_x, mic_y, mic_z, a_broad, a_125, a_250, a_500, a_1k, a_2k, a_4k]` |
| **Y["metrics"]** | `[10]` | `[RT60, DRR_dB, C50_dB, C80_dB, band_RT60_125, ..., band_RT60_4k]` |
| **Y["rir"]** | `[max_len]` | Padded / truncated RIR waveform (32 000 samples @ 16 kHz = 2 s) |
| **Y["edc"]** | `[max_len]` | Schroeder backward-integration Energy Decay Curve (dB) |
| **Y["rir_length"]** | scalar | Original sample count before pad/truncate |

### Pipeline Flow

```
HuggingFace AudioFolder --> decode audio --> pad/truncate --> RIR tensor
           |                                                      |
   metadata CSV (hf_hub_download)                          Schroeder EDC
           |                                                      |
     parse JSON fields --> build X[16] ----------------> (X, Y) Dataset
```

In [1]:
# ============================================================
# Cell 2 — Install Dependencies (Colab)
# ============================================================
!pip install -q datasets huggingface_hub torch pandas numpy scipy

In [2]:
# ============================================================
# Cell 3 — Imports & Constants
# ============================================================
from __future__ import annotations

import ast
import json
import os
import re
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset, Audio
from huggingface_hub import hf_hub_download

# --- Constants ---
OCTAVE_BANDS: List[str] = ["125", "250", "500", "1000", "2000", "4000"]
DEFAULT_MAX_RIR_LEN: int = 32_000          # 2 s @ 16 kHz
INPUT_DIM: int = 16                        # 3 room + 3 src + 3 mic + 1 broadband abs + 6 band abs
METRICS_DIM: int = 10                      # 4 scalar metrics + 6 band RT60s

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cpu


In [3]:
# ============================================================
# Cell 4 — Helper Functions
# ============================================================

def _parse_json_field(val):
    """Robustly parse a CSV cell that stores JSON / Python-literal data."""
    if isinstance(val, (dict, list)):
        return val
    if pd.isna(val) or val is None:
        return None
    try:
        return json.loads(val.replace("'", '"'))
    except (json.JSONDecodeError, AttributeError):
        pass
    try:
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        return None


def compute_edc(rir: np.ndarray) -> np.ndarray:
    """Schroeder backward-integration Energy Decay Curve (dB).

    EDC(t) = integral from t to infinity of h^2(tau) dtau
    Normalised so EDC[0] = 0 dB.
    """
    h2 = rir.astype(np.float64) ** 2
    edc = np.cumsum(h2[::-1])[::-1].copy()
    edc /= edc[0] + 1e-30                    # normalise
    edc_db = 10.0 * np.log10(edc + 1e-30)
    return edc_db.astype(np.float32)


def _extract_sample_id(path: str) -> Optional[str]:
    """Pull 'rir_NNNNNN' out of an audio file path."""
    if not path:
        return None
    m = re.search(r"(rir_\d+)", os.path.basename(str(path)))
    return m.group(1) if m else None


def compute_multiband_edc(
    rir: np.ndarray,
    sr: int = 16_000,
    bands: List[str] = None,
    num_time_steps: int = 256,
) -> np.ndarray:
    """Compute multiband Energy Decay Curves from a broadband RIR.

    Filters the RIR into octave bands using 4th-order Butterworth
    bandpass filters, computes Schroeder EDC per band, then
    downsamples to ``num_time_steps``.

    Parameters
    ----------
    rir : ndarray[L]
        Raw RIR waveform.
    sr : int
        Sample rate in Hz.
    bands : list[str]
        Octave-band centre frequencies as strings (default: OCTAVE_BANDS).
    num_time_steps : int
        Temporal resolution of the output EDC.

    Returns
    -------
    edc_mb : ndarray[num_time_steps, num_bands]
        Multiband EDC in dB.
    """
    from scipy.signal import butter, sosfilt

    if bands is None:
        bands = OCTAVE_BANDS

    nyq = sr / 2.0
    L = len(rir)
    edcs = []

    for fc_str in bands:
        fc = float(fc_str)
        lo = fc / np.sqrt(2.0)
        hi = fc * np.sqrt(2.0)
        # Clamp to Nyquist
        lo = max(lo, 20.0)
        hi = min(hi, nyq - 1.0)

        sos = butter(4, [lo / nyq, hi / nyq], btype="band", output="sos")
        filtered = sosfilt(sos, rir).astype(np.float32)
        edc_band = compute_edc(filtered)
        edcs.append(edc_band)

    # Stack → [L, num_bands]
    edc_full = np.stack(edcs, axis=1)

    # Downsample from L → num_time_steps via strided indexing
    indices = np.linspace(0, L - 1, num_time_steps, dtype=int)
    return edc_full[indices]  # [num_time_steps, num_bands]


def downsample_edc_tensor(
    edc: torch.Tensor,
    num_steps: int,
) -> torch.Tensor:
    """Downsample EDC tensor [B, L] → [B, num_steps] via adaptive avg pooling."""
    return torch.nn.functional.adaptive_avg_pool1d(
        edc.unsqueeze(1), num_steps
    ).squeeze(1)


print("Helper functions defined.")

Helper functions defined.


In [ ]:
# ============================================================
# Cell 5 — RIRMegaDataset Class
# ============================================================

class RIRMegaDataset(Dataset):
    """PyTorch Dataset for the RIRmega corpus.

    Parameters
    ----------
    split : str
        One of "train", "val" (or "validation"), "test".
    max_rir_len : int
        Fixed waveform length in samples (pad / truncate).
    cache_dir : str | None
        Optional HuggingFace cache directory.
    """

    _HF_SPLIT_MAP = {
        "train": "train", "validation": "val",
        "test": "test", "val": "val",
    }

    def __init__(
        self,
        split: str = "train",
        max_rir_len: int = DEFAULT_MAX_RIR_LEN,
        num_time_steps: int = 256,
        sample_rate: int = 16_000,
        cache_dir: Optional[str] = None,
    ) -> None:
        super().__init__()
        self.split = split
        self.max_rir_len = max_rir_len
        self.num_time_steps = num_time_steps
        self.sample_rate = sample_rate

        hf_split = "validation" if split == "val" else split

        # ---- 1. Load HF audio dataset -----------------------------------
        print(f"[RIRMegaDataset] Loading HF audio (split='{hf_split}') …")
        self._hf_ds = load_dataset(
            "mandipgoswami/rirmega",
            split=hf_split,
            trust_remote_code=True,
            cache_dir=cache_dir,
        )

        # ---- 2. Download & parse metadata CSV ---------------------------
        print("[RIRMegaDataset] Downloading metadata CSV …")
        meta_path = hf_hub_download(
            repo_id="mandipgoswami/rirmega",
            filename="metadata/metadata.csv",
            repo_type="dataset",
            cache_dir=cache_dir,
        )
        full_meta = pd.read_csv(meta_path)
        full_meta["id"] = full_meta["id"].astype(str).str.strip()

        # Filter metadata to requested split
        csv_split = self._HF_SPLIT_MAP.get(hf_split, split)
        self._meta = (
            full_meta[full_meta["split"].str.strip() == csv_split]
            .reset_index(drop=True)
        )
        self._meta_id_to_pos = {
            row["id"]: i for i, row in self._meta.iterrows()
        }

        # ---- 3. Build alignment (HF index <-> metadata row) ------------
        self._index_map = self._build_index()
        print(f"[RIRMegaDataset] Ready — {len(self)} aligned samples "
              f"(split='{split}')")

    # ------------------------------------------------------------------
    def _build_index(self) -> List[Tuple[int, int]]:
        """Align HF audio items with metadata rows by sample-ID.
        Falls back to positional matching when paths are unavailable.
        """
        raw = self._hf_ds.cast_column("audio", Audio(decode=False))
        paths = raw["audio"]  # list[dict]

        index_map: List[Tuple[int, int]] = []
        for hf_idx, item in enumerate(paths):
            path = item.get("path", "") if isinstance(item, dict) else ""
            sid = _extract_sample_id(path)
            if sid and sid in self._meta_id_to_pos:
                index_map.append((hf_idx, self._meta_id_to_pos[sid]))

        # Positional fallback
        if not index_map:
            n = min(len(self._hf_ds), len(self._meta))
            index_map = [(i, i) for i in range(n)]

        return index_map

    # ------------------------------------------------------------------
    def __len__(self) -> int:
        return len(self._index_map)

    # ------------------------------------------------------------------
    def _parse_meta_row(self, row: pd.Series) -> Dict:
        """Unpack all JSON-encoded fields for a single metadata row."""
        return {
            "room_size":        _parse_json_field(row["room_size"])       or [0., 0., 0.],
            "source":           _parse_json_field(row["source"])          or [0., 0., 0.],
            "microphone":       _parse_json_field(row["microphone"])      or [0., 0., 0.],
            "absorption":       float(row.get("absorption", 0.0)),
            "absorption_bands": _parse_json_field(row["absorption_bands"]) or {},
            "metrics":          _parse_json_field(row["metrics"])          or {},
        }

    # ------------------------------------------------------------------
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        hf_idx, meta_idx = self._index_map[idx]

        # ---- Audio (decoded on-the-fly by HF datasets) ----------------
        audio = self._hf_ds[hf_idx]["audio"]
        rir_np = np.asarray(audio["array"], dtype=np.float32)
        original_len = len(rir_np)

        # Pad / truncate to fixed length
        if len(rir_np) >= self.max_rir_len:
            rir_np = rir_np[: self.max_rir_len]
        else:
            rir_np = np.pad(rir_np, (0, self.max_rir_len - len(rir_np)))

        # Schroeder EDC (broadband)
        edc_np = compute_edc(rir_np)

        # Multiband EDC target [num_time_steps, num_bands]
        edc_mb_np = compute_multiband_edc(
            rir_np, sr=self.sample_rate,
            num_time_steps=self.num_time_steps,
        )

        # ---- Metadata -------------------------------------------------
        p = self._parse_meta_row(self._meta.iloc[meta_idx])

        band_abs = [float(p["absorption_bands"].get(b, 0.0))
                    for b in OCTAVE_BANDS]

        x = torch.tensor(
            p["room_size"]                          # 3  room dims
            + p["source"]                           # 3  source xyz
            + p["microphone"]                       # 3  mic xyz
            + [p["absorption"]]                     # 1  broadband abs
            + band_abs,                             # 6  band abs
            dtype=torch.float32,
        )  # shape [16]

        m = p["metrics"]
        band_rt60 = m.get("band_rt60s", {})
        metrics_vec = (
            [float(m.get("rt60", 0.0)),
             float(m.get("drr_db", 0.0)),
             float(m.get("c50_db", 0.0)),
             float(m.get("c80_db", 0.0))]
            + [float(band_rt60.get(b, 0.0)) for b in OCTAVE_BANDS]
        )

        y = {
            "metrics":    torch.tensor(metrics_vec, dtype=torch.float32),   # [10]
            "rir":        torch.from_numpy(rir_np),                         # [max_rir_len]
            "edc":        torch.from_numpy(edc_np),                         # [max_rir_len]
            "edc_mb":     torch.from_numpy(edc_mb_np),                      # [num_time_steps, num_bands]
            "rir_length": torch.tensor(original_len, dtype=torch.long),     # scalar
        }

        return x, y


print("RIRMegaDataset class defined.")

SyntaxError: closing parenthesis '}' does not match opening parenthesis '(' on line 149 (574114250.py, line 157)

In [ ]:
# ============================================================
# Cell 6 — Collate Function & DataLoader Factory
# ============================================================

def rir_collate_fn(batch):
    """Stack X tensors and collate Y dicts into batched tensors."""
    xs, ys = zip(*batch)
    return (
        torch.stack(xs, dim=0),
        {
            "metrics":    torch.stack([y["metrics"]    for y in ys], dim=0),
            "rir":        torch.stack([y["rir"]        for y in ys], dim=0),
            "edc":        torch.stack([y["edc"]        for y in ys], dim=0),
            "edc_mb":     torch.stack([y["edc_mb"]     for y in ys], dim=0),
            "rir_length": torch.stack([y["rir_length"] for y in ys], dim=0),
        },
    )


def get_dataloader(
    split: str = "train",
    batch_size: int = 8,
    max_rir_len: int = DEFAULT_MAX_RIR_LEN,
    num_time_steps: int = 256,
    sample_rate: int = 16_000,
    num_workers: int = 2,
    shuffle: Optional[bool] = None,
    cache_dir: Optional[str] = None,
    **loader_kwargs,
) -> DataLoader:
    """Convenience factory — returns a ready-to-iterate DataLoader."""
    if shuffle is None:
        shuffle = split == "train"

    ds = RIRMegaDataset(split=split, max_rir_len=max_rir_len,
                        num_time_steps=num_time_steps,
                        sample_rate=sample_rate,
                        cache_dir=cache_dir)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        collate_fn=rir_collate_fn,
        pin_memory=torch.cuda.is_available(),
        **loader_kwargs,
    )

print("Collate function & DataLoader factory defined.")

In [ ]:
# ============================================================
# Cell 7 — Unit Test: Verify Tensor Shapes
# ============================================================

MAX_LEN = DEFAULT_MAX_RIR_LEN
BATCH = 4

print("=" * 64)
print("  Unit Test — RIRMegaDataset  (Phase 1: Data Pipeline)")
print("=" * 64)

# ---- Single-sample test ----------------------------------------------
ds = RIRMegaDataset(split="train", max_rir_len=MAX_LEN)

x, y = ds[0]
print(f"\n[Single sample — index 0]")
print(f"  X shape      : {x.shape}           (expect [{INPUT_DIM}])")
print(f"  X dtype      : {x.dtype}")
print(f"  X values     : {x.tolist()[:6]} …")
print(f"  Y keys       : {sorted(y.keys())}")
print(f"  metrics shape: {y['metrics'].shape}  (expect [{METRICS_DIM}])")
print(f"  rir shape    : {y['rir'].shape}  (expect [{MAX_LEN}])")
print(f"  edc shape    : {y['edc'].shape}  (expect [{MAX_LEN}])")
print(f"  rir_length   : {y['rir_length'].item()}")

assert x.shape == (INPUT_DIM,),              f"X shape mismatch: {x.shape}"
assert y["metrics"].shape == (METRICS_DIM,), f"metrics shape mismatch: {y['metrics'].shape}"
assert y["rir"].shape == (MAX_LEN,),         f"rir shape mismatch: {y['rir'].shape}"
assert y["edc"].shape == (MAX_LEN,),         f"edc shape mismatch: {y['edc'].shape}"
assert y["rir_length"].dtype == torch.long

# ---- Batch / DataLoader test -----------------------------------------
loader = get_dataloader(
    split="train", batch_size=BATCH,
    max_rir_len=MAX_LEN, num_workers=0,
)
xb, yb = next(iter(loader))

print(f"\n[Batch — batch_size={BATCH}]")
print(f"  X batch      : {xb.shape}        (expect [{BATCH}, {INPUT_DIM}])")
print(f"  metrics batch: {yb['metrics'].shape}  (expect [{BATCH}, {METRICS_DIM}])")
print(f"  rir batch    : {yb['rir'].shape}  (expect [{BATCH}, {MAX_LEN}])")
print(f"  edc batch    : {yb['edc'].shape}  (expect [{BATCH}, {MAX_LEN}])")
print(f"  rir_lengths  : {yb['rir_length'].tolist()}")

assert xb.shape == (BATCH, INPUT_DIM)
assert yb["metrics"].shape == (BATCH, METRICS_DIM)
assert yb["rir"].shape == (BATCH, MAX_LEN)
assert yb["edc"].shape == (BATCH, MAX_LEN)

# ---- GPU transfer sanity check (Colab) --------------------------------
xb_gpu = xb.to(DEVICE)
print(f"\n  X on device  : {xb_gpu.device}")

print(f"\n{'─' * 64}")
print("  ✓  All assertions passed — data pipeline is operational.")
print("=" * 64)

---
# Phase 2 — LSTM Multiband EDC Predictor

**Goal:** Map the 16-d physical input features → a predicted **multiband Energy Decay Curve** (dB) using an LSTM.

### Architecture

```
Input [B, 16]  (room dims, src/mic xyz, absorption)
       │
  ┌────▼────┐
  │   MLP   │  2-layer encoder
  │ Encoder │  → context vector [B, hidden]
  └────┬────┘
       │  projected into h₀, c₀ for each LSTM layer
  ┌────▼────────────────────────┐
  │  LSTM  (T learned temporal  │
  │         input embeddings)   │  num_layers, hidden_dim
  └────┬────────────────────────┘
       │  [B, T, hidden]
  ┌────▼────┐
  │ Linear  │  projection head
  │  Head   │  → [B, T, num_bands]  (EDC in dB)
  └─────────┘
```

| Param | Default | Meaning |
|-------|---------|---------|
| `input_dim` | 16 | Physical features from dataset X |
| `hidden_dim` | 256 | LSTM & encoder width |
| `num_layers` | 2 | Stacked LSTM layers |
| `num_bands` | 6 | Octave bands (125 – 4000 Hz) |
| `num_time_steps` | 256 | EDC time-frames (~7.8 ms each @ 2 s) |

In [ ]:
# ============================================================
# Cell 9 — LSTM Multiband EDC Predictor
# ============================================================
import torch.nn as nn


class MultibandEDCPredictor(nn.Module):
    """LSTM-based predictor: physical room parameters → multiband EDC.

    The model conditions an LSTM on the room/source/absorption features
    via learned initial hidden & cell states, then unrolls over *T*
    time-steps using a learned temporal embedding to produce a
    frequency-dependent Energy Decay Curve.

    Parameters
    ----------
    input_dim : int
        Dimensionality of physics input vector (default 16).
    hidden_dim : int
        Width of the encoder MLP and LSTM hidden state.
    num_layers : int
        Number of stacked LSTM layers.
    num_bands : int
        Number of output frequency bands (default 6 octave bands).
    num_time_steps : int
        Temporal resolution of the predicted EDC.
    dropout : float
        Dropout applied between LSTM layers (ignored when num_layers=1).
    """

    def __init__(
        self,
        input_dim: int = INPUT_DIM,
        hidden_dim: int = 256,
        num_layers: int = 2,
        num_bands: int = len(OCTAVE_BANDS),     # 6
        num_time_steps: int = 256,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.num_bands = num_bands
        self.num_time_steps = num_time_steps

        # ---- MLP encoder: input features → context vector ---------------
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(inplace=True),
        )

        # ---- Project context → LSTM initial states ----------------------
        #   h0 and c0 each have shape [num_layers, B, hidden_dim]
        self.h0_proj = nn.Linear(hidden_dim, num_layers * hidden_dim)
        self.c0_proj = nn.Linear(hidden_dim, num_layers * hidden_dim)

        # ---- Learned temporal input embeddings --------------------------
        #   One embedding vector per time-step, shared across the batch
        self.time_embed = nn.Parameter(
            torch.randn(1, num_time_steps, hidden_dim) * 0.02
        )

        # ---- LSTM -------------------------------------------------------
        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        # ---- Output projection head -------------------------------------
        #   Maps each LSTM time-step output → num_bands EDC values (dB)
        self.output_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim // 2, num_bands),
        )

        self._init_weights()

    # ------------------------------------------------------------------
    def _init_weights(self) -> None:
        """Xavier init for linear layers; orthogonal for LSTM weights."""
        for name, param in self.named_parameters():
            if "lstm" in name and "weight" in name:
                nn.init.orthogonal_(param)
            elif "lstm" in name and "bias" in name:
                nn.init.zeros_(param)
                # Set forget-gate bias to 1 for better gradient flow
                hidden = self.hidden_dim
                param.data[hidden : 2 * hidden].fill_(1.0)
            elif param.dim() >= 2 and "time_embed" not in name:
                nn.init.xavier_uniform_(param)

    # ------------------------------------------------------------------
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        x : Tensor[B, input_dim]
            Physical input features from the dataset.

        Returns
        -------
        edc_pred : Tensor[B, num_time_steps, num_bands]
            Predicted multiband Energy Decay Curve (dB scale).
        """
        B = x.size(0)

        # 1. Encode input features
        ctx = self.encoder(x)                          # [B, hidden]

        # 2. Derive LSTM initial states from context
        h0 = (
            self.h0_proj(ctx)                          # [B, num_layers*hidden]
            .view(B, self.num_layers, self.hidden_dim)
            .permute(1, 0, 2)                          # [num_layers, B, hidden]
            .contiguous()
        )
        c0 = (
            self.c0_proj(ctx)
            .view(B, self.num_layers, self.hidden_dim)
            .permute(1, 0, 2)
            .contiguous()
        )

        # 3. Expand temporal embeddings to batch → LSTM input
        lstm_in = self.time_embed.expand(B, -1, -1)   # [B, T, hidden]

        # 4. LSTM forward pass
        lstm_out, _ = self.lstm(lstm_in, (h0, c0))    # [B, T, hidden]

        # 5. Project to multiband EDC
        edc_pred = self.output_head(lstm_out)          # [B, T, num_bands]

        return edc_pred

    # ------------------------------------------------------------------
    def predict_broadband(self, x: torch.Tensor) -> torch.Tensor:
        """Convenience: average predicted band EDCs → single broadband EDC.

        Returns
        -------
        edc_broadband : Tensor[B, num_time_steps]
        """
        return self.forward(x).mean(dim=-1)

    # ------------------------------------------------------------------
    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


print("MultibandEDCPredictor class defined.")

In [ ]:
# ============================================================
# Cell 10 — Unit Test: LSTM Model Shape Verification
# ============================================================

BATCH_TEST = 4
NUM_BANDS = len(OCTAVE_BANDS)   # 6
T_STEPS = 256

print("=" * 64)
print("  Unit Test — MultibandEDCPredictor  (Phase 2: LSTM Model)")
print("=" * 64)

# ---- Instantiate model ------------------------------------------------
model = MultibandEDCPredictor(
    input_dim=INPUT_DIM,
    hidden_dim=256,
    num_layers=2,
    num_bands=NUM_BANDS,
    num_time_steps=T_STEPS,
    dropout=0.1,
).to(DEVICE)

print(f"\n  Model parameters : {model.count_parameters():,}")
print(f"  Device           : {DEVICE}")

# ---- Forward pass with dummy data ------------------------------------
dummy_x = torch.randn(BATCH_TEST, INPUT_DIM, device=DEVICE)
with torch.no_grad():
    edc_pred = model(dummy_x)

print(f"\n[Forward pass — dummy batch]")
print(f"  Input  X shape   : {dummy_x.shape}       (expect [{BATCH_TEST}, {INPUT_DIM}])")
print(f"  Output EDC shape : {edc_pred.shape}  (expect [{BATCH_TEST}, {T_STEPS}, {NUM_BANDS}])")
print(f"  Output dtype     : {edc_pred.dtype}")
print(f"  Output range     : [{edc_pred.min().item():.4f}, {edc_pred.max().item():.4f}]")

assert edc_pred.shape == (BATCH_TEST, T_STEPS, NUM_BANDS), \
    f"Shape mismatch: {edc_pred.shape}"
assert edc_pred.device.type == DEVICE.type

# ---- Broadband convenience method ------------------------------------
with torch.no_grad():
    edc_broad = model.predict_broadband(dummy_x)

print(f"\n[Broadband EDC]")
print(f"  Shape            : {edc_broad.shape}      (expect [{BATCH_TEST}, {T_STEPS}])")

assert edc_broad.shape == (BATCH_TEST, T_STEPS)

# ---- Gradient flow check ---------------------------------------------
model.train()
x_grad = torch.randn(BATCH_TEST, INPUT_DIM, device=DEVICE, requires_grad=True)
pred = model(x_grad)
loss = pred.sum()
loss.backward()

grad_ok = x_grad.grad is not None and x_grad.grad.abs().sum().item() > 0
print(f"\n  Gradients flow   : {'✓ OK' if grad_ok else '✗ BROKEN'}")
assert grad_ok, "Gradients do not flow back to input!"

# ---- Integration with Phase 1 dataset (if loaded) --------------------
try:
    sample_x, _ = ds[0]               # from Phase 1 unit test
    sample_x = sample_x.unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        real_pred = model(sample_x)
    print(f"\n[Real sample from dataset]")
    print(f"  Input  X         : {sample_x.shape}")
    print(f"  Output EDC       : {real_pred.shape}")
    print(f"  EDC[0, 0, :]     : {real_pred[0, 0, :].tolist()}")  # first time-step, all bands
except NameError:
    print("\n  (Skipping real-sample test — run Phase 1 cells first)")

print(f"\n{'─' * 64}")
print("  ✓  All assertions passed — LSTM model is operational.")
print("=" * 64)

---
# Phase 3 — Custom Loss Functions (Physics-Informed)

**Goal:** Combine a standard EDC reconstruction loss with physics-informed regularization terms derived from the fundamental equations of linear room acoustics.

---

### 1. EDC Reconstruction Loss

Standard root-mean-square error between predicted and target multiband EDC:

$$\mathcal{L}_{\text{EDC}} = \sqrt{\frac{1}{B \cdot T \cdot F}\sum_{b,t,f}\bigl(\hat{E}_{b,t,f} - E_{b,t,f}\bigr)^2}$$

---

### 2. Acoustic Continuity Equation (Mass Conservation)

In lossless linear acoustics the continuity equation relates pressure $p$ and particle velocity $\mathbf{u}$:

$$\frac{\partial p}{\partial t} + \rho_0 c^2 \nabla \cdot \mathbf{u} = 0$$

We penalise the residual so the network learns spatially consistent fields:

$$\mathcal{L}_{\text{cont}} = \frac{1}{N}\sum_i\left(\frac{\partial p_i}{\partial t} + \rho_0 c^2 \left(\frac{\partial u_x}{\partial x} + \frac{\partial u_y}{\partial y} + \frac{\partial u_z}{\partial z}\right)_i\right)^2$$

---

### 3. Linearized Momentum Equation (Euler's Equation)

Newton's second law for an inviscid fluid:

$$\rho_0 \frac{\partial \mathbf{u}}{\partial t} + \nabla p = 0$$

Residual computed **per spatial component**:

$$\mathcal{L}_{\text{mom}} = \frac{1}{N}\sum_i\left\|\rho_0\frac{\partial \mathbf{u}_i}{\partial t} + \nabla p_i\right\|^2$$

---

### Combined Loss

$$\mathcal{L} = \mathcal{L}_{\text{EDC}} + \lambda_{\text{cont}}\,\mathcal{L}_{\text{cont}} + \lambda_{\text{mom}}\,\mathcal{L}_{\text{mom}}$$

| Hyper-param | Default | Meaning |
|-------------|---------|---------|
| `lambda_cont` | 0.01 | Weight for continuity residual |
| `lambda_mom` | 0.01 | Weight for momentum residual |
| `rho0` | 1.225 | Air density (kg/m³) |
| `c` | 343.0 | Speed of sound (m/s) |

In [ ]:
# ============================================================
# Cell 12 — Physics-Informed Loss Functions
# ============================================================
import torch
import torch.nn as nn
from typing import Dict, Optional


# ------------------------------------------------------------------
#  1.  EDC Reconstruction Loss (RMSE)
# ------------------------------------------------------------------

class EDCReconstructionLoss(nn.Module):
    """Root-mean-square error between predicted and target EDC (dB).

    Supports both multiband [B, T, F] and broadband [B, T] tensors.
    """

    def forward(
        self,
        edc_pred: torch.Tensor,
        edc_target: torch.Tensor,
    ) -> torch.Tensor:
        return torch.sqrt(torch.mean((edc_pred - edc_target) ** 2) + 1e-8)


# ------------------------------------------------------------------
#  2.  Acoustic Continuity Equation Residual
# ------------------------------------------------------------------

def continuity_residual(
    pressure: torch.Tensor,
    velocity: torch.Tensor,
    coords: torch.Tensor,
    time: torch.Tensor,
    rho0: float = 1.225,
    c: float = 343.0,
) -> torch.Tensor:
    """Residual of the acoustic continuity equation.

    ∂p/∂t  +  ρ₀ c² ∇·u  =  0

    All inputs must have `requires_grad=True` so that
    `torch.autograd.grad` can compute the necessary derivatives.

    Parameters
    ----------
    pressure : Tensor[N, 1]
        Acoustic pressure field at N collocation points.
    velocity : Tensor[N, 3]
        Particle velocity (u_x, u_y, u_z) at the same points.
    coords : Tensor[N, 3]
        Spatial coordinates (x, y, z) of each collocation point.
        Must have `requires_grad=True`.
    time : Tensor[N, 1]
        Time coordinate for each point.
        Must have `requires_grad=True`.
    rho0 : float
        Equilibrium air density (kg/m³).
    c : float
        Speed of sound (m/s).

    Returns
    -------
    residual : Tensor[N, 1]
        Point-wise residual of the continuity equation.
    """
    # ∂p/∂t  — temporal derivative of pressure
    dp_dt = torch.autograd.grad(
        outputs=pressure,
        inputs=time,
        grad_outputs=torch.ones_like(pressure),
        create_graph=True,
        retain_graph=True,
    )[0]                                                # [N, 1]

    # ∂u_x/∂x, ∂u_y/∂y, ∂u_z/∂z  — spatial divergence of velocity
    div_u = torch.zeros_like(pressure)                  # [N, 1]
    for dim in range(3):
        du_i = torch.autograd.grad(
            outputs=velocity[:, dim : dim + 1],
            inputs=coords,
            grad_outputs=torch.ones_like(pressure),
            create_graph=True,
            retain_graph=True,
        )[0][:, dim : dim + 1]                          # ∂u_i/∂x_i
        div_u = div_u + du_i

    return dp_dt + rho0 * c ** 2 * div_u                # [N, 1]


# ------------------------------------------------------------------
#  3.  Linearized Momentum Equation Residual (Euler's Equation)
# ------------------------------------------------------------------

def momentum_residual(
    pressure: torch.Tensor,
    velocity: torch.Tensor,
    coords: torch.Tensor,
    time: torch.Tensor,
    rho0: float = 1.225,
) -> torch.Tensor:
    """Residual of the linearized momentum equation.

    ρ₀ ∂u/∂t  +  ∇p  =  0

    Parameters
    ----------
    pressure : Tensor[N, 1]
        Acoustic pressure at N collocation points.
    velocity : Tensor[N, 3]
        Particle velocity (u_x, u_y, u_z).
    coords : Tensor[N, 3]
        Spatial coordinates (requires_grad=True).
    time : Tensor[N, 1]
        Temporal coordinate (requires_grad=True).
    rho0 : float
        Equilibrium air density.

    Returns
    -------
    residual : Tensor[N, 3]
        Per-component residual of the momentum equation.
    """
    # ∇p  — spatial gradient of pressure  [N, 3]
    grad_p = torch.autograd.grad(
        outputs=pressure,
        inputs=coords,
        grad_outputs=torch.ones_like(pressure),
        create_graph=True,
        retain_graph=True,
    )[0]                                                # [N, 3]

    # ∂u/∂t  — temporal derivative of each velocity component  [N, 3]
    du_dt = torch.autograd.grad(
        outputs=velocity,
        inputs=time,
        grad_outputs=torch.ones_like(velocity),
        create_graph=True,
        retain_graph=True,
    )[0]                                                # [N, 3]

    return rho0 * du_dt + grad_p                        # [N, 3]


# ------------------------------------------------------------------
#  4.  Combined Physics-Informed Loss
# ------------------------------------------------------------------

class PhysicsInformedRIRLoss(nn.Module):
    """Unified loss = L_edc  +  λ_cont · L_cont  +  λ_mom · L_mom.

    The EDC term is always computed.  The physics residual terms are
    only added when the caller supplies the collocation-point fields
    (pressure, velocity, coords, time).  This makes the loss usable
    both for **pure EDC regression** (early training) and for
    **physics-regularised fine-tuning** (later phases).

    Parameters
    ----------
    lambda_cont : float
        Weight for the continuity-equation residual.
    lambda_mom : float
        Weight for the momentum-equation residual.
    rho0 : float
        Equilibrium air density (kg/m³).
    c : float
        Speed of sound (m/s).
    """

    def __init__(
        self,
        lambda_cont: float = 0.01,
        lambda_mom: float = 0.01,
        rho0: float = 1.225,
        c: float = 343.0,
    ) -> None:
        super().__init__()
        self.lambda_cont = lambda_cont
        self.lambda_mom = lambda_mom
        self.rho0 = rho0
        self.c = c
        self.edc_loss_fn = EDCReconstructionLoss()

    def forward(
        self,
        edc_pred: torch.Tensor,
        edc_target: torch.Tensor,
        pressure: Optional[torch.Tensor] = None,
        velocity: Optional[torch.Tensor] = None,
        coords: Optional[torch.Tensor] = None,
        time: Optional[torch.Tensor] = None,
    ) -> Dict[str, torch.Tensor]:
        """
        Parameters
        ----------
        edc_pred : Tensor[B, T, F] or [B, T]
            Predicted EDC from the LSTM model.
        edc_target : Tensor — same shape as edc_pred
            Ground-truth EDC from the dataset.
        pressure : Tensor[N, 1], optional
            Acoustic pressure at N collocation points.
        velocity : Tensor[N, 3], optional
            Particle velocity at the same points.
        coords : Tensor[N, 3], optional  (requires_grad=True)
            Spatial coordinates of collocation points.
        time : Tensor[N, 1], optional    (requires_grad=True)
            Temporal coordinates of collocation points.

        Returns
        -------
        losses : dict
            "total"     — combined scalar loss (for .backward())
            "edc"       — EDC RMSE component
            "continuity"— continuity residual MSE  (0 if not provided)
            "momentum"  — momentum residual MSE    (0 if not provided)
        """
        device = edc_pred.device

        # ---- 1. EDC reconstruction loss --------------------------------
        l_edc = self.edc_loss_fn(edc_pred, edc_target)

        # ---- 2. Physics residuals (optional) ---------------------------
        has_physics = all(
            t is not None for t in [pressure, velocity, coords, time]
        )

        if has_physics:
            # Continuity: ∂p/∂t + ρ₀c² ∇·u = 0
            r_cont = continuity_residual(
                pressure, velocity, coords, time,
                rho0=self.rho0, c=self.c,
            )
            l_cont = torch.mean(r_cont ** 2)

            # Momentum: ρ₀ ∂u/∂t + ∇p = 0
            r_mom = momentum_residual(
                pressure, velocity, coords, time,
                rho0=self.rho0,
            )
            l_mom = torch.mean(r_mom ** 2)
        else:
            l_cont = torch.tensor(0.0, device=device)
            l_mom = torch.tensor(0.0, device=device)

        # ---- 3. Combine ------------------------------------------------
        total = l_edc + self.lambda_cont * l_cont + self.lambda_mom * l_mom

        return {
            "total": total,
            "edc": l_edc,
            "continuity": l_cont,
            "momentum": l_mom,
        }


print("PhysicsInformedRIRLoss and residual functions defined.")

In [ ]:
# ============================================================
# Cell 13 — Unit Test: Physics-Informed Loss Functions
# ============================================================

BATCH_TEST = 4
T_STEPS = 256
NUM_BANDS = len(OCTAVE_BANDS)   # 6
N_COLLOC = 128                   # collocation points for PDE test

print("=" * 64)
print("  Unit Test — PhysicsInformedRIRLoss  (Phase 3: Loss Functions)")
print("=" * 64)

# ---- 1. Pure EDC loss (no physics terms) -----------------------------
criterion = PhysicsInformedRIRLoss(
    lambda_cont=0.01, lambda_mom=0.01,
    rho0=1.225, c=343.0,
).to(DEVICE)

edc_pred   = torch.randn(BATCH_TEST, T_STEPS, NUM_BANDS, device=DEVICE)
edc_target = torch.randn(BATCH_TEST, T_STEPS, NUM_BANDS, device=DEVICE)

losses = criterion(edc_pred, edc_target)

print(f"\n[EDC-only mode — no physics terms]")
print(f"  total        : {losses['total'].item():.6f}")
print(f"  edc (RMSE)   : {losses['edc'].item():.6f}")
print(f"  continuity   : {losses['continuity'].item():.6f}  (expect 0)")
print(f"  momentum     : {losses['momentum'].item():.6f}  (expect 0)")

assert losses["continuity"].item() == 0.0, "Continuity should be 0 when not supplied"
assert losses["momentum"].item()   == 0.0, "Momentum should be 0 when not supplied"
assert losses["total"].item() > 0.0,       "Total loss should be > 0"
assert torch.isclose(losses["total"], losses["edc"]), \
    "Total should equal EDC when physics terms are absent"

# ---- 2. Standalone continuity residual test --------------------------
print(f"\n[Continuity residual — synthetic collocation points]")

coords = torch.randn(N_COLLOC, 3, device=DEVICE, requires_grad=True)
time   = torch.randn(N_COLLOC, 1, device=DEVICE, requires_grad=True)

# Build synthetic P(x,y,z,t) and u(x,y,z,t) as differentiable
# functions of coords & time so autograd can compute derivatives.
#   p = sin(x + y + z + t)   →  dp/dt = cos(...)
#   u = [sin(t), sin(t), sin(t)]  →  ∂u_x/∂x=0, etc, du/dt = cos(t)
pressure = torch.sin(coords.sum(dim=1, keepdim=True) + time)  # [N,1]
velocity = torch.cat([torch.sin(time)] * 3, dim=1)            # [N,3]

r_cont = continuity_residual(pressure, velocity, coords, time)
print(f"  Residual shape : {r_cont.shape}  (expect [{N_COLLOC}, 1])")
print(f"  Residual range : [{r_cont.min().item():.4f}, {r_cont.max().item():.4f}]")
assert r_cont.shape == (N_COLLOC, 1)

# Gradient should flow back
l_cont = torch.mean(r_cont ** 2)
l_cont.backward(retain_graph=True)
assert coords.grad is not None, "Gradients must flow to coords"
print(f"  Grad flows ✓")

# ---- 3. Standalone momentum residual test ----------------------------
print(f"\n[Momentum residual — synthetic collocation points]")

coords2 = torch.randn(N_COLLOC, 3, device=DEVICE, requires_grad=True)
time2   = torch.randn(N_COLLOC, 1, device=DEVICE, requires_grad=True)

pressure2 = torch.sin(coords2.sum(dim=1, keepdim=True) + time2)
velocity2 = torch.cat([torch.sin(time2)] * 3, dim=1)

r_mom = momentum_residual(pressure2, velocity2, coords2, time2)
print(f"  Residual shape : {r_mom.shape}  (expect [{N_COLLOC}, 3])")
print(f"  Residual range : [{r_mom.min().item():.4f}, {r_mom.max().item():.4f}]")
assert r_mom.shape == (N_COLLOC, 3)

l_mom = torch.mean(r_mom ** 2)
l_mom.backward(retain_graph=True)
assert coords2.grad is not None, "Gradients must flow to coords"
print(f"  Grad flows ✓")

# ---- 4. Full combined loss with physics terms ------------------------
print(f"\n[Full physics-informed loss]")

coords3 = torch.randn(N_COLLOC, 3, device=DEVICE, requires_grad=True)
time3   = torch.randn(N_COLLOC, 1, device=DEVICE, requires_grad=True)
p3 = torch.sin(coords3.sum(dim=1, keepdim=True) + time3)
v3 = torch.cat([torch.sin(time3)] * 3, dim=1)

full_losses = criterion(
    edc_pred, edc_target,
    pressure=p3, velocity=v3, coords=coords3, time=time3,
)

print(f"  total        : {full_losses['total'].item():.6f}")
print(f"  edc (RMSE)   : {full_losses['edc'].item():.6f}")
print(f"  continuity   : {full_losses['continuity'].item():.6f}")
print(f"  momentum     : {full_losses['momentum'].item():.6f}")

assert full_losses["total"] > full_losses["edc"], \
    "Total should exceed pure EDC when physics terms are active"
assert full_losses["continuity"].item() > 0
assert full_losses["momentum"].item() > 0

# Backward through the full combined loss
full_losses["total"].backward()
assert coords3.grad is not None
print(f"  Full backward ✓")

# ---- 5. Integration with Phase 2 model (if loaded) ------------------
try:
    model.eval()
    dummy_x = torch.randn(BATCH_TEST, INPUT_DIM, device=DEVICE)
    with torch.no_grad():
        pred = model(dummy_x)
    # Use random targets to simulate ground-truth
    target = torch.randn_like(pred)
    integration_losses = criterion(pred, target)
    print(f"\n[Integration: LSTM output → Loss]")
    print(f"  pred shape   : {pred.shape}")
    print(f"  EDC loss     : {integration_losses['edc'].item():.6f}")
except NameError:
    print("\n  (Skipping integration — run Phase 2 cells first)")

print(f"\n{'─' * 64}")
print("  ✓  All assertions passed — loss functions are operational.")
print("=" * 64)

---
# Phase 4 — Differentiable Feedback Delay Network (FDN)

**Goal:** Model the late reverberation tail with a 16-channel Feedback Delay Network whose parameters are **fully differentiable**, enabling end-to-end gradient-based optimisation.

---

### FDN Signal Flow

```
input x[n] ──► [b_0 ... b_15] gain ──┐
                                      │
              ┌───────────────────────▼───────────────────────┐
              │        16 parallel delay lines                │
              │   z^{-m_0}, z^{-m_1}, ..., z^{-m_15}         │
              │        (learnable log-spaced delays)          │
              └───────────────┬───────────────────────────────┘
                              │
                       ┌──────▼──────┐
                       │  16x16      │
                       │ Feedback    │ ◄── Hadamard / householder
                       │  Matrix A   │     (unitary, lossless)
                       └──────┬──────┘
                              │
              ┌───────────────▼───────────────────────────────┐
              │   per-channel absorption filter                │
              │   g_i = alpha_i · exp(-beta_i · m_i / fs)     │
              │        (frequency-independent decay)           │
              └───────────────┬───────────────────────────────┘
                              │
                       feedback ──► back to delay input
                              │
              ┌───────────────▼──────┐
              │  [c_0 ... c_15] gain │ ──► output y[n]
              └──────────────────────┘
```

### Learnable Parameters

| Parameter | Shape | Constraint | Meaning |
|-----------|:-----:|:-----------|:--------|
| `log_delays` | `[16]` | Sigmoid → [0, 50 ms] | Delay-line lengths in log-space |
| `alpha` | `[16]` | Sigmoid → (0, 1) | Per-channel decay rate |
| `beta` | `[16]` | Softplus → > 0 | Per-channel absorption scale |
| `input_gains` | `[16]` | Unconstrained | Input bus gains $b_i$ |
| `output_gains` | `[16]` | Unconstrained | Output bus gains $c_i$ |

In [ ]:
# ============================================================
# Cell 15 — Differentiable Feedback Delay Network (FDN)
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import Optional


def _hadamard_matrix(n: int) -> torch.Tensor:
    """Construct a normalised Hadamard matrix of order *n* (must be power of 2).

    The Hadamard matrix is orthogonal / unitary, guaranteeing lossless
    energy circulation in the feedback loop.
    """
    assert n > 0 and (n & (n - 1)) == 0, "n must be a power of 2"
    H = torch.tensor([[1.0]])
    while H.size(0) < n:
        H = torch.cat([
            torch.cat([H,  H], dim=1),
            torch.cat([H, -H], dim=1),
        ], dim=0)
    return H / math.sqrt(n)


class DifferentiableFDN(nn.Module):
    """Feedback Delay Network with fully differentiable parameters.

    The module synthesises the **late reverberation tail** of a Room
    Impulse Response by circulating energy through *N* parallel delay
    lines connected by a unitary feedback matrix.

    All timing and decay parameters live in unconstrained spaces and are
    mapped to their physical ranges via smooth, differentiable
    activations so that standard gradient-based optimisers can tune them.

    Parameters
    ----------
    num_delays : int
        Number of parallel feedback loops (default 16).
    max_delay_ms : float
        Upper bound for delay-line lengths in milliseconds (default 50).
    sample_rate : int
        Audio sample rate in Hz (default 16000).
    output_length : int
        Number of output samples to synthesise (default 32000 = 2 s).
    """

    def __init__(
        self,
        num_delays: int = 16,
        max_delay_ms: float = 50.0,
        sample_rate: int = 16_000,
        output_length: int = DEFAULT_MAX_RIR_LEN,   # 32 000
    ) -> None:
        super().__init__()
        self.num_delays = num_delays
        self.max_delay_ms = max_delay_ms
        self.sample_rate = sample_rate
        self.output_length = output_length

        # Maximum delay in samples
        self.max_delay_samples = int(max_delay_ms * sample_rate / 1000.0)

        # ---- Learnable parameters (unconstrained) ----------------------

        # log_delays: mapped via sigmoid → (0, max_delay_samples)
        #   Initialised so that delays span the log range evenly
        init_delays = torch.linspace(-2.0, 2.0, num_delays)
        self.log_delays = nn.Parameter(init_delays)

        # alpha: per-channel decay rate, sigmoid → (0, 1)
        #   Init near 0.95 (long reverb)
        self.alpha_raw = nn.Parameter(torch.full((num_delays,), 2.94))  # sigmoid(2.94) ≈ 0.95

        # beta: per-channel absorption scale, softplus → (0, ∞)
        #   Controls how fast energy decays per round-trip
        self.beta_raw = nn.Parameter(torch.full((num_delays,), 0.5))

        # Input / output bus gains
        self.input_gains = nn.Parameter(torch.ones(num_delays) / math.sqrt(num_delays))
        self.output_gains = nn.Parameter(torch.ones(num_delays) / math.sqrt(num_delays))

        # ---- Fixed feedback matrix (Hadamard, unitary) -----------------
        self.register_buffer("feedback_matrix", _hadamard_matrix(num_delays))

    # ------------------------------------------------------------------
    #  Constrained parameter accessors
    # ------------------------------------------------------------------

    @property
    def delays_samples(self) -> torch.Tensor:
        """Differentiable delay lengths in samples (float).

        Sigmoid maps log_delays → (0, 1), then scaled to
        (1, max_delay_samples).  Kept as float for differentiability;
        fractional delays are handled via linear interpolation.
        """
        return 1.0 + torch.sigmoid(self.log_delays) * (self.max_delay_samples - 1)

    @property
    def alpha(self) -> torch.Tensor:
        """Per-channel decay rate ∈ (0, 1)."""
        return torch.sigmoid(self.alpha_raw)

    @property
    def beta(self) -> torch.Tensor:
        """Per-channel absorption scale > 0."""
        return F.softplus(self.beta_raw)

    @property
    def decay_gains(self) -> torch.Tensor:
        """Per-channel gain applied each round-trip:
            g_i = alpha_i · exp(-beta_i · m_i / fs)
        """
        m = self.delays_samples                   # [N]
        return self.alpha * torch.exp(-self.beta * m / self.sample_rate)

    # ------------------------------------------------------------------
    #  Fractional delay via linear interpolation
    # ------------------------------------------------------------------

    @staticmethod
    def _frac_delay_read(
        buffer: torch.Tensor,
        write_pos: int,
        delay: torch.Tensor,
        max_len: int,
    ) -> torch.Tensor:
        """Read from a circular buffer at a fractional delay position.

        Uses linear interpolation between the two nearest integer taps
        to keep the operation differentiable.

        Parameters
        ----------
        buffer : Tensor[B, N, max_buf]
            Circular delay-line buffer.
        write_pos : int
            Current write pointer (samples).
        delay : Tensor[N]
            Delay lengths in samples (float, may be fractional).
        max_len : int
            Buffer length.

        Returns
        -------
        out : Tensor[B, N]
        """
        # Integer and fractional parts
        delay_int = delay.long()                          # floor
        frac = delay - delay_int.float()                  # fractional part

        idx0 = (write_pos - delay_int) % max_len          # [N]
        idx1 = (write_pos - delay_int - 1) % max_len      # [N]

        # Gather from buffer:  buffer[:, i, idx[i]]
        B = buffer.size(0)
        idx0_exp = idx0.unsqueeze(0).expand(B, -1)        # [B, N]
        idx1_exp = idx1.unsqueeze(0).expand(B, -1)

        val0 = buffer.gather(2, idx0_exp.unsqueeze(2)).squeeze(2)  # [B, N]
        val1 = buffer.gather(2, idx1_exp.unsqueeze(2)).squeeze(2)

        return (1.0 - frac.unsqueeze(0)) * val0 + frac.unsqueeze(0) * val1

    # ------------------------------------------------------------------
    #  Forward — synthesise the late reverb tail
    # ------------------------------------------------------------------

    def forward(
        self,
        impulse: Optional[torch.Tensor] = None,
        batch_size: int = 1,
    ) -> torch.Tensor:
        """Synthesise a late-reverberation impulse response.

        Parameters
        ----------
        impulse : Tensor[B, output_length], optional
            Excitation signal fed into the FDN.  If *None*, a unit
            impulse (delta at sample 0) is used.
        batch_size : int
            Used only when ``impulse`` is None.

        Returns
        -------
        y : Tensor[B, output_length]
            Synthesised late-reverberation tail.
        """
        device = self.log_delays.device
        N = self.num_delays
        T = self.output_length
        buf_len = self.max_delay_samples + 2  # extra sample for interp

        if impulse is not None:
            B = impulse.size(0)
        else:
            B = batch_size
            impulse = torch.zeros(B, T, device=device)
            impulse[:, 0] = 1.0                           # unit impulse

        # Circular delay-line buffer  [B, N, buf_len]
        buf = torch.zeros(B, N, buf_len, device=device)
        output = torch.zeros(B, T, device=device)

        delays = self.delays_samples                      # [N]
        g = self.decay_gains                              # [N]
        A = self.feedback_matrix                          # [N, N]
        b_in = self.input_gains                           # [N]
        c_out = self.output_gains                         # [N]

        for t in range(T):
            # 1. Read from delay lines (fractional)
            dl_out = self._frac_delay_read(buf, t, delays, buf_len)  # [B, N]

            # 2. Apply per-channel decay
            dl_out = dl_out * g.unsqueeze(0)              # [B, N]

            # 3. Mix through feedback matrix
            fb = torch.matmul(dl_out, A.T)                # [B, N]

            # 4. Inject input
            x_in = impulse[:, t].unsqueeze(1) * b_in.unsqueeze(0)  # [B, N]
            dl_in = fb + x_in

            # 5. Write into delay lines
            w_idx = t % buf_len
            buf[:, :, w_idx] = dl_in

            # 6. Tap output
            output[:, t] = (dl_out * c_out.unsqueeze(0)).sum(dim=1)

        return output

    # ------------------------------------------------------------------
    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def get_delay_info(self) -> dict:
        """Return a snapshot of the constrained delay parameters."""
        with torch.no_grad():
            ms = self.delays_samples * 1000.0 / self.sample_rate
            return {
                "delays_ms": ms.cpu().tolist(),
                "delays_samples": self.delays_samples.cpu().tolist(),
                "alpha": self.alpha.cpu().tolist(),
                "beta": self.beta.cpu().tolist(),
                "decay_gains": self.decay_gains.cpu().tolist(),
            }


print("DifferentiableFDN class defined.")
print(f"  Hadamard 16x16 norm check: {_hadamard_matrix(16).norm():.4f} (expect 1.0)")

In [ ]:
# ============================================================
# Cell 16 — Unit Test: Differentiable FDN
# ============================================================

print("=" * 64)
print("  Unit Test — DifferentiableFDN  (Phase 4: FDN)")
print("=" * 64)

# ---- 1. Instantiate ----------------------------------------------------
fdn = DifferentiableFDN(
    num_delays=16,
    max_delay_ms=50.0,
    sample_rate=16_000,
    output_length=DEFAULT_MAX_RIR_LEN,    # 32 000
).to(DEVICE)

print(f"\n  Learnable parameters : {fdn.count_parameters()}")
print(f"  Device               : {DEVICE}")
print(f"  Max delay (samples)  : {fdn.max_delay_samples}")

# ---- 2. Delay constraint verification ----------------------------------
info = fdn.get_delay_info()
delays_ms = info["delays_ms"]
print(f"\n[Delay constraints]")
print(f"  delays (ms)  : {[f'{d:.2f}' for d in delays_ms]}")
print(f"  range        : [{min(delays_ms):.2f}, {max(delays_ms):.2f}] ms")
print(f"  all in (0, 50): {all(0 < d < 50 for d in delays_ms)}")
assert all(0 < d < 50 for d in delays_ms), "Delays must be in (0, 50) ms"

print(f"  alpha range  : [{min(info['alpha']):.4f}, {max(info['alpha']):.4f}]  (expect (0,1))")
assert all(0 < a < 1 for a in info["alpha"]), "Alpha must be in (0, 1)"

print(f"  beta range   : [{min(info['beta']):.4f}, {max(info['beta']):.4f}]  (expect > 0)")
assert all(b > 0 for b in info["beta"]), "Beta must be > 0"

print(f"  decay gains  : [{min(info['decay_gains']):.4f}, {max(info['decay_gains']):.4f}]")

# ---- 3. Forward pass — unit impulse -----------------------------------
BATCH_FDN = 2

with torch.no_grad():
    y = fdn(batch_size=BATCH_FDN)

print(f"\n[Forward — unit impulse]")
print(f"  Output shape : {y.shape}  (expect [{BATCH_FDN}, {DEFAULT_MAX_RIR_LEN}])")
print(f"  Output range : [{y.min().item():.6f}, {y.max().item():.6f}]")
print(f"  Energy (RMS) : {y.pow(2).mean().sqrt().item():.6f}")

assert y.shape == (BATCH_FDN, DEFAULT_MAX_RIR_LEN), f"Shape mismatch: {y.shape}"
assert not torch.isnan(y).any(), "NaN detected in output"
assert not torch.isinf(y).any(), "Inf detected in output"

# ---- 4. Forward pass — custom excitation signal -------------------------
excitation = torch.randn(BATCH_FDN, DEFAULT_MAX_RIR_LEN, device=DEVICE) * 0.01
with torch.no_grad():
    y_exc = fdn(impulse=excitation)

print(f"\n[Forward — noise excitation]")
print(f"  Output shape : {y_exc.shape}")
print(f"  Output range : [{y_exc.min().item():.6f}, {y_exc.max().item():.6f}]")
assert y_exc.shape == (BATCH_FDN, DEFAULT_MAX_RIR_LEN)

# ---- 5. Gradient flow — can we backprop through the FDN? ---------------
print(f"\n[Gradient flow check]")
fdn.train()

# Shorter output for faster gradient test
fdn_short = DifferentiableFDN(
    num_delays=16, max_delay_ms=50.0,
    sample_rate=16_000, output_length=1000,   # 1000 samples for speed
).to(DEVICE)

y_grad = fdn_short(batch_size=1)
loss = y_grad.pow(2).sum()
loss.backward()

grads_ok = True
for name, p in fdn_short.named_parameters():
    if p.grad is None:
        print(f"  WARNING: {name} has no gradient!")
        grads_ok = False
    else:
        g_norm = p.grad.norm().item()
        print(f"  {name:20s}  grad norm = {g_norm:.6f}")

assert grads_ok, "Some parameters did not receive gradients!"

# ---- 6. Feedback matrix is unitary -------------------------------------
A = fdn.feedback_matrix
ident_err = (A @ A.T - torch.eye(16, device=DEVICE)).abs().max().item()
print(f"\n[Feedback matrix unitarity]")
print(f"  ||A A^T - I||_inf   : {ident_err:.2e}  (expect ~0)")
assert ident_err < 1e-5, f"Feedback matrix not unitary: error = {ident_err}"

# ---- 7. Integration with Phase 2 LSTM (if loaded) ---------------------
try:
    model.eval()
    fdn.eval()
    dummy_x = torch.randn(1, INPUT_DIM, device=DEVICE)
    with torch.no_grad():
        edc_pred = model(dummy_x)               # [1, 256, 6]
        rir_late = fdn(batch_size=1)             # [1, 32000]
    print(f"\n[Integration: LSTM → FDN]")
    print(f"  EDC pred shape  : {edc_pred.shape}")
    print(f"  Late reverb shape: {rir_late.shape}")
except NameError:
    print("\n  (Skipping integration — run Phase 2 cells first)")

print(f"\n{'─' * 64}")
print("  ✓  All assertions passed — Differentiable FDN is operational.")
print("=" * 64)

---
# Phase 5 — Training Loop

**Goal:** Wire together all preceding phases into an end-to-end training pipeline.

### Data Flow (Training)

```
RIRMegaDataset                   MultibandEDCPredictor
  X [B, 16]  ──────────────────►  forward(X) ──► edc_pred [B, 256, 6]
  Y["edc_mb"] [B, 256, 6]  ──┐                        │
                              └──► PhysicsInformedRIRLoss ──► L_total
                                     │
                              L_edc (RMSE)  +  λ_cont·L_cont  +  λ_mom·L_mom
```

### Training Strategy

| Stage | Epochs | What | Loss |
|:------|:------:|:-----|:-----|
| **Warm-up** | 1 – 20 | LSTM EDC predictor only | EDC RMSE (physics terms off: `lambda=0`) |
| **Physics** | 21 – 50 | LSTM + optional physics | EDC RMSE + continuity + momentum (when collocation points are supplied) |
| **FDN** | Optional | Differentiable FDN fine-tuning | RIR waveform or EDC match against target late tail |

### Hyperparameters

| Param | Default | Notes |
|-------|---------|-------|
| `epochs` | 50 | Adjust for Colab GPU budget |
| `batch_size` | 8 | Increase to 16/32 on large-VRAM GPUs |
| `lr` | 1e-3 | Adam learning rate |
| `weight_decay` | 1e-5 | L2 regularisation |
| `grad_clip` | 1.0 | Max gradient norm |
| `log_every` | 100 | Print loss every N steps |
| `scheduler` | ReduceLROnPlateau | patience=5, factor=0.5 |

In [ ]:
# ============================================================
# Cell 18 — Training Configuration & Trainer
# ============================================================
import time as _time
from dataclasses import dataclass


@dataclass
class TrainingConfig:
    """All hyper-parameters in one place."""

    # ---- Data ----------------------------------------------------------
    batch_size: int = 8
    num_workers: int = 2
    max_rir_len: int = DEFAULT_MAX_RIR_LEN        # 32 000
    sample_rate: int = 16_000

    # ---- LSTM Model ----------------------------------------------------
    hidden_dim: int = 256
    num_layers: int = 2
    num_time_steps: int = 256
    num_bands: int = len(OCTAVE_BANDS)             # 6
    model_dropout: float = 0.1

    # ---- FDN (optional) ------------------------------------------------
    train_fdn: bool = False
    fdn_num_delays: int = 16
    fdn_max_delay_ms: float = 50.0
    fdn_output_length: int = 4_000                 # shortened for speed
    fdn_weight: float = 0.1                        # weight for FDN loss

    # ---- Loss ----------------------------------------------------------
    lambda_cont: float = 0.0                       # off during warm-up
    lambda_mom: float = 0.0

    # ---- Optimiser -----------------------------------------------------
    lr: float = 1e-3
    weight_decay: float = 1e-5
    grad_clip: float = 1.0

    # ---- Scheduler -----------------------------------------------------
    scheduler_patience: int = 5
    scheduler_factor: float = 0.5

    # ---- Training ------------------------------------------------------
    epochs: int = 50
    log_every: int = 100                           # steps between prints
    val_every_epoch: int = 1


# ======================================================================
#  RIR Trainer
# ======================================================================

class RIRTrainer:
    """End-to-end training harness for the Physics-Informed RIR pipeline.

    Brings together:
      - Phase 1  RIRMegaDataset / DataLoader
      - Phase 2  MultibandEDCPredictor (LSTM)
      - Phase 3  PhysicsInformedRIRLoss
      - Phase 4  DifferentiableFDN (optional)
    """

    def __init__(self, cfg: TrainingConfig) -> None:
        self.cfg = cfg
        self.device = DEVICE

        # ---- Build components ------------------------------------------
        self._build_dataloaders()
        self._build_models()
        self._build_criterion()
        self._build_optimiser()

        # ---- Bookkeeping -----------------------------------------------
        self.global_step = 0
        self.best_val_loss = float("inf")
        self.history: Dict[str, list] = {
            "train_loss": [], "train_edc": [],
            "val_loss": [], "val_edc": [],
        }

    # ------------------------------------------------------------------
    #  Construction helpers
    # ------------------------------------------------------------------

    def _build_dataloaders(self) -> None:
        c = self.cfg
        self.train_loader = get_dataloader(
            split="train", batch_size=c.batch_size,
            max_rir_len=c.max_rir_len,
            num_time_steps=c.num_time_steps,
            sample_rate=c.sample_rate,
            num_workers=c.num_workers, shuffle=True,
        )
        self.val_loader = get_dataloader(
            split="val", batch_size=c.batch_size,
            max_rir_len=c.max_rir_len,
            num_time_steps=c.num_time_steps,
            sample_rate=c.sample_rate,
            num_workers=c.num_workers, shuffle=False,
        )
        print(f"[Trainer] Train batches: {len(self.train_loader)}, "
              f"Val batches: {len(self.val_loader)}")

    def _build_models(self) -> None:
        c = self.cfg

        self.lstm = MultibandEDCPredictor(
            input_dim=INPUT_DIM,
            hidden_dim=c.hidden_dim,
            num_layers=c.num_layers,
            num_bands=c.num_bands,
            num_time_steps=c.num_time_steps,
            dropout=c.model_dropout,
        ).to(self.device)
        print(f"[Trainer] LSTM params: {self.lstm.count_parameters():,}")

        if c.train_fdn:
            self.fdn = DifferentiableFDN(
                num_delays=c.fdn_num_delays,
                max_delay_ms=c.fdn_max_delay_ms,
                sample_rate=c.sample_rate,
                output_length=c.fdn_output_length,
            ).to(self.device)
            print(f"[Trainer] FDN  params: {self.fdn.count_parameters():,}")
        else:
            self.fdn = None

    def _build_criterion(self) -> None:
        self.criterion = PhysicsInformedRIRLoss(
            lambda_cont=self.cfg.lambda_cont,
            lambda_mom=self.cfg.lambda_mom,
        ).to(self.device)

    def _build_optimiser(self) -> None:
        c = self.cfg
        params = list(self.lstm.parameters())
        if self.fdn is not None:
            params += list(self.fdn.parameters())

        self.optimiser = torch.optim.Adam(
            params, lr=c.lr, weight_decay=c.weight_decay,
        )
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimiser,
            mode="min",
            factor=c.scheduler_factor,
            patience=c.scheduler_patience,
            verbose=True,
        )

    # ------------------------------------------------------------------
    #  Training & validation
    # ------------------------------------------------------------------

    def train_one_epoch(self, epoch: int) -> Dict[str, float]:
        """Train for one full pass over the training set."""
        self.lstm.train()
        if self.fdn is not None:
            self.fdn.train()

        running = {"total": 0.0, "edc": 0.0, "n": 0}

        for batch_idx, (x_batch, y_batch) in enumerate(self.train_loader):
            x = x_batch.to(self.device)                                # [B, 16]
            edc_target = y_batch["edc_mb"].to(self.device)             # [B, T, F]

            # ---- Forward -----------------------------------------------
            edc_pred = self.lstm(x)                                    # [B, T, F]

            # ---- Loss --------------------------------------------------
            losses = self.criterion(edc_pred, edc_target)

            # Optional FDN loss: compare FDN output EDC with target
            if self.fdn is not None:
                fdn_rir = self.fdn(batch_size=x.size(0))               # [B, fdn_len]
                # Compute EDC of FDN output and compare to broadband target
                fdn_edc = torch.cumsum(fdn_rir.flip(1).pow(2), dim=1).flip(1)
                fdn_edc = fdn_edc / (fdn_edc[:, :1] + 1e-30)
                fdn_edc_db = 10.0 * torch.log10(fdn_edc + 1e-30)

                # Downsample target broadband EDC to FDN length
                edc_broad = y_batch["edc"].to(self.device)             # [B, max_rir_len]
                edc_broad_ds = downsample_edc_tensor(
                    edc_broad, self.cfg.fdn_output_length
                )
                fdn_loss = torch.sqrt(
                    torch.mean((fdn_edc_db - edc_broad_ds) ** 2) + 1e-8
                )
                losses["total"] = losses["total"] + self.cfg.fdn_weight * fdn_loss
                losses["fdn"] = fdn_loss

            # ---- Backward ----------------------------------------------
            self.optimiser.zero_grad()
            losses["total"].backward()

            # Gradient clipping
            if self.cfg.grad_clip > 0:
                nn.utils.clip_grad_norm_(
                    self.lstm.parameters(), self.cfg.grad_clip
                )
                if self.fdn is not None:
                    nn.utils.clip_grad_norm_(
                        self.fdn.parameters(), self.cfg.grad_clip
                    )

            self.optimiser.step()

            # ---- Logging -----------------------------------------------
            bs = x.size(0)
            running["total"] += losses["total"].item() * bs
            running["edc"] += losses["edc"].item() * bs
            running["n"] += bs
            self.global_step += 1

            if self.global_step % self.cfg.log_every == 0:
                avg_t = running["total"] / running["n"]
                avg_e = running["edc"] / running["n"]
                lr_now = self.optimiser.param_groups[0]["lr"]
                msg = (f"  [Epoch {epoch+1} | Step {self.global_step:>6d}]  "
                       f"loss={avg_t:.4f}  edc={avg_e:.4f}  lr={lr_now:.2e}")
                if self.fdn is not None and "fdn" in losses:
                    msg += f"  fdn={losses['fdn'].item():.4f}"
                print(msg)

        n = running["n"]
        return {
            "total": running["total"] / max(n, 1),
            "edc": running["edc"] / max(n, 1),
        }

    # ------------------------------------------------------------------
    @torch.no_grad()
    def validate(self) -> Dict[str, float]:
        """Run one full pass over the validation set."""
        self.lstm.eval()
        if self.fdn is not None:
            self.fdn.eval()

        running = {"total": 0.0, "edc": 0.0, "n": 0}

        for x_batch, y_batch in self.val_loader:
            x = x_batch.to(self.device)
            edc_target = y_batch["edc_mb"].to(self.device)

            edc_pred = self.lstm(x)
            losses = self.criterion(edc_pred, edc_target)

            bs = x.size(0)
            running["total"] += losses["total"].item() * bs
            running["edc"] += losses["edc"].item() * bs
            running["n"] += bs

        n = running["n"]
        return {
            "total": running["total"] / max(n, 1),
            "edc": running["edc"] / max(n, 1),
        }

    # ------------------------------------------------------------------
    def fit(self) -> Dict[str, list]:
        """Full training loop over all epochs."""
        print("=" * 64)
        print("  Training — Physics-Informed RIR Pipeline")
        print("=" * 64)
        print(f"  Device     : {self.device}")
        print(f"  Epochs     : {self.cfg.epochs}")
        print(f"  Batch size : {self.cfg.batch_size}")
        print(f"  LR         : {self.cfg.lr}")
        print(f"  FDN        : {'ON' if self.fdn is not None else 'OFF'}")
        print(f"  Physics λ  : cont={self.cfg.lambda_cont}, "
              f"mom={self.cfg.lambda_mom}")
        print("=" * 64)

        for epoch in range(self.cfg.epochs):
            t0 = _time.time()

            # ---- Train -------------------------------------------------
            train_metrics = self.train_one_epoch(epoch)
            self.history["train_loss"].append(train_metrics["total"])
            self.history["train_edc"].append(train_metrics["edc"])

            # ---- Validate ----------------------------------------------
            if (epoch + 1) % self.cfg.val_every_epoch == 0:
                val_metrics = self.validate()
                self.history["val_loss"].append(val_metrics["total"])
                self.history["val_edc"].append(val_metrics["edc"])

                # LR scheduler step
                self.scheduler.step(val_metrics["total"])

                # Best model checkpoint
                if val_metrics["total"] < self.best_val_loss:
                    self.best_val_loss = val_metrics["total"]
                    torch.save(self.lstm.state_dict(), "best_lstm.pt")
                    if self.fdn is not None:
                        torch.save(self.fdn.state_dict(), "best_fdn.pt")
                    marker = " ★ saved"
                else:
                    marker = ""
            else:
                val_metrics = {"total": float("nan"), "edc": float("nan")}
                marker = ""

            elapsed = _time.time() - t0
            print(
                f"Epoch {epoch+1:>3d}/{self.cfg.epochs}  "
                f"train_loss={train_metrics['total']:.4f}  "
                f"val_loss={val_metrics['total']:.4f}  "
                f"({elapsed:.1f}s){marker}"
            )

        print(f"\n{'─' * 64}")
        print(f"  Training complete.  Best val loss: {self.best_val_loss:.4f}")
        print("=" * 64)
        return self.history


print("TrainingConfig and RIRTrainer defined.")

In [ ]:
# ============================================================
# Cell 19 — Launch Training
# ============================================================
# Toggle DRY_RUN = True for a quick smoke-test with synthetic data
# (no HuggingFace download needed).  Set False for real training.
DRY_RUN = True

if DRY_RUN:
    # ------------------------------------------------------------------
    #  Dry-run: validate full pipeline with synthetic tensors
    # ------------------------------------------------------------------
    print("=" * 64)
    print("  DRY RUN — Smoke Test (synthetic data, 2 steps)")
    print("=" * 64)

    B, T, F = 4, 256, len(OCTAVE_BANDS)

    # Synthetic batch
    syn_x = torch.randn(B, INPUT_DIM, device=DEVICE)
    syn_edc_mb = torch.randn(B, T, F, device=DEVICE) * -30.0   # dB-scale
    syn_edc_broad = torch.randn(B, DEFAULT_MAX_RIR_LEN, device=DEVICE)

    # LSTM
    lstm_dry = MultibandEDCPredictor(
        input_dim=INPUT_DIM, hidden_dim=128, num_layers=1,
        num_bands=F, num_time_steps=T, dropout=0.0,
    ).to(DEVICE)

    # Loss
    criterion_dry = PhysicsInformedRIRLoss(
        lambda_cont=0.0, lambda_mom=0.0,
    ).to(DEVICE)

    # Optimiser
    opt_dry = torch.optim.Adam(lstm_dry.parameters(), lr=1e-3)

    # Two training steps
    for step in range(2):
        lstm_dry.train()
        pred = lstm_dry(syn_x)                          # [B, T, F]
        losses = criterion_dry(pred, syn_edc_mb)
        opt_dry.zero_grad()
        losses["total"].backward()
        nn.utils.clip_grad_norm_(lstm_dry.parameters(), 1.0)
        opt_dry.step()
        print(f"  Step {step+1}:  loss={losses['total'].item():.4f}  "
              f"edc={losses['edc'].item():.4f}")

    # FDN forward + backward (short output)
    fdn_dry = DifferentiableFDN(
        num_delays=16, max_delay_ms=50.0,
        sample_rate=16_000, output_length=500,
    ).to(DEVICE)
    fdn_out = fdn_dry(batch_size=1)
    fdn_loss = fdn_out.pow(2).mean()
    fdn_loss.backward()
    print(f"  FDN step :  fdn_loss={fdn_loss.item():.6f}")

    # Shape summary
    print(f"\n  LSTM input  : {syn_x.shape}")
    print(f"  LSTM output : {pred.shape}")
    print(f"  FDN output  : {fdn_out.shape}")
    print(f"  Loss keys   : {sorted(losses.keys())}")

    print(f"\n{'─' * 64}")
    print("  ✓  Dry run passed — full pipeline is end-to-end functional.")
    print("=" * 64)
    print("\n  ➜  Set DRY_RUN = False and re-run this cell to train on")
    print("     the real RIRmega dataset.")

else:
    # ------------------------------------------------------------------
    #  Real training on RIRmega dataset
    # ------------------------------------------------------------------
    cfg = TrainingConfig(
        # Data
        batch_size=8,
        num_workers=2,
        # Model
        hidden_dim=256,
        num_layers=2,
        num_time_steps=256,
        model_dropout=0.1,
        # FDN (disabled by default — enable for Phase 4 fine-tuning)
        train_fdn=False,
        # Loss (physics terms off for warm-up; increase after ~20 epochs)
        lambda_cont=0.0,
        lambda_mom=0.0,
        # Optimiser
        lr=1e-3,
        weight_decay=1e-5,
        grad_clip=1.0,
        # Schedule
        epochs=50,
        log_every=100,
    )
    trainer = RIRTrainer(cfg)
    history = trainer.fit()